# 02 - Train + benchmark (run with **Accelerator: GPU T4**, internet ON)

Attach as inputs: (1) the `tinygpt.zip` dataset, (2) the **output of notebook 01** (tokenizer + shards), and in later sessions (3) the **output of the previous run of this notebook** so training can resume.

Set `STAGE` below, then **Save Version -> Save & Run All (Commit)** so it runs in the background (no browser needed).

| STAGE | what it does | rough time (estimate - check the tok/s in the log) |
|---|---|---|
| `check` | tests + GPU smoke test + throughput probe | ~10 min |
| `target` | trains the 28M target (auto-stops before the session limit; run again to resume) | hours |
| `drafts` | trains draft_4m, draft_2l, and the distilled draft | ~1-3 h |
| `bench` | evaluation + benchmarks E1/E2/E3/E5 + analysis + plots | ~30-60 min |

In [ ]:
STAGE = 'check'          # 'check' | 'target' | 'drafts' | 'bench'
TIME_LIMIT_MIN = 660     # stop gracefully before the 12 h session limit (checkpoint saved; re-run to resume)

In [ ]:
import glob, os, shutil, subprocess, sys

def sh(cmd):
    """Run a shell command and stream its output live; raise if it fails."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")

def setup_code():
    """Copy the tinygpt code (uploaded as a Kaggle Dataset) into /kaggle/working and install it."""
    work = "/kaggle/working/tinygpt"
    if not os.path.exists(work):
        hits = glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)
        assert hits, "Code not found. Upload tinygpt.zip as a Kaggle Dataset and attach it (Add Data)."
        shutil.copytree(os.path.dirname(hits[0]), work)
    os.chdir(work)
    sh("pip install -q -e . regex pyyaml matplotlib")
    return work

work = setup_code()
sh('nvidia-smi')

In [ ]:
# locate data + restore earlier progress (attached outputs of previous runs)
meta = glob.glob('/kaggle/input/**/shards/meta.json', recursive=True)
assert meta, 'Attach the output of notebook 01 (it contains shards/meta.json)'
DATA = os.path.dirname(os.path.dirname(meta[0]))          # .../tinygpt_data
print('data dir:', DATA)
OVR = f'--set train.data_dir={DATA}/shards --set train.tokenizer={DATA}/tok/tokenizer.json'
for f in glob.glob('/kaggle/input/**/runs/*/*', recursive=True):
    if os.path.isfile(f) and f.split('.')[-1] in ('pt', 'csv', 'json'):
        rel = f.split('/runs/', 1)[1]
        dst = os.path.join(work, 'runs', rel)
        if not os.path.exists(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy(f, dst); print('restored', rel)

In [ ]:
if STAGE == 'check':
    sh('python -m pytest -q -x tests/test_model.py tests/test_kv_cache.py tests/test_tokenizer.py')
    sh('python scripts/smoke_test.py --device cuda')          # full tiny pipeline on the GPU
    sh('python scripts/probe_xpu.py cuda')
    # 2-minute throughput probe of the real config: note tok/s, then decide if the full run fits your quota
    sh(f'python -m tinygpt.train --config configs/target_28m.yaml --compile --max-steps 60 --set train.out_dir=runs/_probe {OVR} --set train.eval_interval=1000')
    # optional LR sweep (about 15 min): sh(f'python scripts/lr_sweep.py --config configs/target_28m.yaml --steps 300 --set train.data_dir={DATA}/shards --set train.tokenizer={DATA}/tok/tokenizer.json')

In [ ]:
if STAGE == 'target':
    sh(f'python -m tinygpt.train --config configs/target_28m.yaml --compile --resume --time-limit-minutes {TIME_LIMIT_MIN} {OVR}')

In [ ]:
if STAGE == 'drafts':
    for cfg in ('draft_4m', 'draft_2l'):
        sh(f'python -m tinygpt.train --config configs/{cfg}.yaml --compile --resume --time-limit-minutes {TIME_LIMIT_MIN} {OVR}')
    # distilled draft needs the finished target (runs/target_28m/model.pt, restored above)
    assert os.path.exists('runs/target_28m/model.pt'), 'attach the output of the finished target run'
    sh(f'python -m tinygpt.train --config configs/draft_4m_distilled.yaml --compile --resume --time-limit-minutes {TIME_LIMIT_MIN} {OVR}')

In [ ]:
if STAGE == 'bench':
    D = f'--data-dir {DATA}/shards'
    sh(f'python -m tinygpt.eval.perplexity --ckpt runs/target_28m/model.pt {D} --dtype fp16')
    sh(f'python -m tinygpt.eval.perplexity --ckpt runs/draft_4m/model.pt {D} --dtype fp16')
    sh(f'python -m tinygpt.eval.samples --ckpt runs/target_28m/model.pt --tokenizer {DATA}/tok/tokenizer.json {D} --out results/samples/target_28m.md')
    T = 'runs/target_28m/model.pt'
    sh(f'python -m bench.run E1 --target {T} {D} --power-mode kaggle')
    sh(f'python -m bench.run E2 --target {T} {D} --power-mode kaggle')
    drafts = 'draft_2l=runs/draft_2l/model.pt,draft_4l=runs/draft_4m/model.pt,draft_4l_distilled=runs/draft_4m_distilled/model.pt'
    sh(f'python -m bench.run E3 --target {T} --drafts {drafts} {D} --power-mode kaggle')
    sh(f'python -m bench.run E5 --target {T} --draft runs/draft_4m/model.pt {D}')
    sh('python -m bench.analyze --results results && python -m bench.plot --results results')
    sh(f'python -m export.to_hf_llama --ckpt {T} --tok {DATA}/tok/tokenizer.json --out export/hf')   # needs: pip install transformers tokenizers

In [ ]:
# what will be saved when you commit the notebook
sh('ls -la runs/*/ 2>/dev/null | head -60; du -sh /kaggle/working')